# AgriNav -- Phase-2 detector: **2-epoch pilot** (Colab / GPU)

**This is not a training run.** It is two 2-epoch runs whose only purpose is to
produce the evidence for four decisions that would otherwise ride along silently
inside a 90-minute 18-epoch run. `docs/GATE_STATUS.md` marks the full run
**NO-GO until this pilot reports**.

| What is read | What it decides |
|---|---|
| `grad_norm/{p50,p90,p99,clipped_fraction}` | the real value of `grad_clip`, or whether to drop it |
| `parity/max_conf_ratio` per epoch | whether the train/eval BatchNorm gap is opening, and from which epoch |
| `train/cls_loss_pos` vs `train/cls_loss_neg` | whether the classifier is finding objects or just learning "background" |
| `bn/eval_mode`, `bn/grad_off` | whether the BN freeze actually held -- observed, not requested |

**Why two arms.** The RiceSEG arm freezes **57 of 58** BN layers, the ImageNet
arm **48 of 58**. They differ because they loaded different things, not because
the policy is inconsistent -- `--bn-freeze-scope` asserts each expected set and
aborts on a partial load. Running only one arm cannot separate "this backbone is
bad" from "this BN configuration is bad".

**Why the numbers here are small.** 2 epochs, and the diagnostics are observation
only: the parity probe runs under `no_grad` and restores every BN buffer, so
nothing in this notebook changes what training does.

This notebook only *drives* `agrinav.training.weeddet_train` and
`agrinav.training.pilot_report`. It defines no model, loss, metric, or training
logic of its own.

## 1. Confirm GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code (pinned checkout)

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/Bmerrysmith/Autonomous-tractor-system.git'
# Pinned. A mutable branch name means two runs of "the same" notebook can execute
# different code, and it silently did once already: the 2026-07-28 phase-2 runs
# cloned `master`, which had no --val-ann-file, and selected checkpoints on
# TRAINING loss while the notebook's comments claimed otherwise.
# Set REPO_SHA = None to deliberately take the tip of REPO_REF instead.
REPO_REF  = 'feat/training-observability'
REPO_SHA  = 'abb42ab33d5b4823ae1de5daa7c55af72571fe9d'
REPO_DIR  = '/content/agrinav'

# Capabilities this notebook requires from the checkout. Each must exist in the
# trainer; a stale clone fails HERE rather than after both arms have burned GPU
# time producing a metrics.jsonl with none of the columns the readout needs.
REQUIRED_FLAGS = ('riceseg-backbone', 'val-ann-file', 'val-images-root',
                  'bn-policy', 'bn-freeze-scope', 'dump-grad-norms',
                  'parity-probe-images', 'overfit-min-ap50')

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
print('GitHub token found in Colab secrets:', bool(token))

def _redact(s):
    return s.replace(token, '***') if token else s

# GIT_TERMINAL_PROMPT=0 turns an auth failure into an immediate error, not a hang.
env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

def _git(*args, check=True):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], env=env, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f'git {args[0]} failed:\n' + _redact(r.stderr))
    return r.stdout.strip()

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--branch', REPO_REF, auth_url, REPO_DIR],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            'CLONE FAILED -- stopping so later cells do not cascade.\n\n'
            + _redact(r.stderr) +
            '\nFix: add a GITHUB_TOKEN Colab secret (fine-grained PAT, Contents:Read), '
            'or make the repo public, or upload the repo to Drive and set REPO_DIR.')
    _git('remote', 'set-url', 'origin', REPO_URL)
else:
    _git('remote', 'set-url', 'origin', auth_url)
    _git('fetch', 'origin', '--prune')
    _git('remote', 'set-url', 'origin', REPO_URL)

if REPO_SHA:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--depth', '50', auth_url, REPO_SHA],
                   env=env, capture_output=True, text=True)
    _git('checkout', '--force', REPO_SHA)
else:
    _git('checkout', '--force', f'origin/{REPO_REF}')

os.chdir(REPO_DIR)
HEAD_SHA = _git('rev-parse', 'HEAD')
if REPO_SHA and HEAD_SHA != REPO_SHA:
    raise SystemExit(f'checkout landed on {HEAD_SHA}, expected {REPO_SHA}')

trainer = 'src/agrinav/training/weeddet_train.py'
if not os.path.exists(trainer):
    raise SystemExit(f'checkout at {HEAD_SHA} has no {trainer} -- wrong ref?')
with open(trainer, encoding='utf-8') as fh:
    trainer_src = fh.read()
absent = [flag for flag in REQUIRED_FLAGS if flag not in trainer_src]
if absent:
    raise SystemExit(
        f'this checkout ({HEAD_SHA[:8]}) is missing required trainer flag(s): {absent}.\n'
        'Delete /content/agrinav and re-run, or update REPO_REF/REPO_SHA to a commit that '
        'has them. Running anyway would produce a pilot that answers nothing.')

readout = 'src/agrinav/training/pilot_report.py'
if not os.path.exists(readout):
    raise SystemExit(
        f'checkout at {HEAD_SHA[:8]} has no {readout} -- the readout cell would fail '
        'after both arms had already run. Repin to a commit that has it.')

print(f'repo ready at {REPO_DIR} @ {HEAD_SHA}')
print('  required flags present:', ', '.join(REQUIRED_FLAGS))

## 4. Install the package

In [ ]:
import subprocess

# check=True: a failed install must stop the notebook. `get_ipython().system(...)`
# discards the exit code, so a broken environment used to sail on into training.
subprocess.run(['pip', 'install', '-q', '-e', '.[train]'], check=True)
import agrinav
print('agrinav', agrinav.__version__, 'installed from', os.getcwd())

## 5. Extract the curated RICE data + locate the backbone

Same archive, same sha256 gate, and the same member-by-member rejection of any
`test/` path as the full-run notebook. The pilot is short, but it is still a real
run on the real split -- a diagnostic computed on the wrong data answers nothing.

In [ ]:
import glob, hashlib, json, zipfile
from pathlib import PurePosixPath

DRIVE_P2 = '/content/drive/MyDrive/agrinav_data/rice_phase2_v2'
LOCAL    = '/content/rice_curated'

# The rebuilt archive, and the one it replaces. The 2026-07-27
# `RICE_curated_phase2.zip` is CONTAMINATED -- it mis-exported 231 of the 261
# intended sealed-test images into train/valid and omitted 233 intended
# train/valid images (it dropped the native test/ folder instead of applying
# grouped_split.json). Do not train on it.
ARCHIVE = 'RICE_phase2_rebuild.zip'
BANNED_ARCHIVES = {'RICE_curated_phase2.zip'}
EXPECTED_SHA256 = '40eb6370f41eeb53333918cfbeb55d3696a848067e2c96a389a8e1508be3fd03'

zip_path = f'{DRIVE_P2}/{ARCHIVE}'
if not os.path.exists(zip_path):
    hits = [p for p in glob.glob(f'/content/drive/MyDrive/**/{ARCHIVE}', recursive=True)]
    assert hits, f'{ARCHIVE} not found in Drive. See the phase-2 notebook for how to build it.'
    zip_path = hits[0]
assert os.path.basename(zip_path) not in BANNED_ARCHIVES, (
    f'{os.path.basename(zip_path)} is the contaminated archive. See {DRIVE_P2}/GATE_STATUS.md.')

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

ARCHIVE_SHA256 = _sha256(zip_path)
print('zip   :', zip_path)
print('sha256:', ARCHIVE_SHA256)
assert EXPECTED_SHA256 is None or ARCHIVE_SHA256 == EXPECTED_SHA256, (
    f'archive sha256 mismatch.\n  expected {EXPECTED_SHA256}\n  got      {ARCHIVE_SHA256}')

def _safe_members(names):
    """Reject traversal/absolute paths and any member under a `test` directory."""
    for name in names:
        if name.endswith('/'):
            continue
        parts = PurePosixPath(name).parts
        assert not PurePosixPath(name).is_absolute() and not name.startswith('/'), \
            f'absolute path in archive: {name}'
        assert '..' not in parts, f'path traversal in archive: {name}'
        assert 'test' not in parts, (
            f'archive contains a test-split member ({name}). The test split must stay '
            'physically out of any archive used for training.')
    return True

if not os.path.isdir(f'{LOCAL}/images/train'):
    os.makedirs(LOCAL, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        _safe_members(z.namelist())
        z.extractall(LOCAL)
else:
    print('reusing existing extraction at', LOCAL, '-- delete it if the archive changed')

TRAIN_JSON = f'{LOCAL}/annotations/instances_train.coco.json'
VAL_JSON   = f'{LOCAL}/annotations/instances_valid.coco.json'
TRAIN_IMGS = f'{LOCAL}/images/train'
VAL_IMGS   = f'{LOCAL}/images/valid'
for p in (TRAIN_JSON, VAL_JSON, TRAIN_IMGS, VAL_IMGS):
    assert os.path.exists(p), f'missing after extract: {p}'

subprocess.run(
    ['python', '-B', '-u', '-m', 'agrinav.data.build_rice_phase2', 'preflight',
     '--out-root', LOCAL],
    check=True)

for name, j in (('train', TRAIN_JSON), ('valid', VAL_JSON)):
    d = json.load(open(j))
    print(f'{name}: {len(d["images"])} imgs  {len(d["annotations"])} anns')

BACKBONE = f'{DRIVE_P2}/riceseg_backbone.pth'
if not os.path.exists(BACKBONE):
    BACKBONE = '/content/drive/MyDrive/agrinav_data/out/riceseg_backbone.pth'
assert os.path.exists(BACKBONE), (
    f'phase-1 backbone not found at {BACKBONE}. Run the RiceSEG pretraining notebook first.')
print('backbone:', BACKBONE)

## 6. Wiring gate: decoded detections on 8 memorised images

This gate no longer passes on `final_loss < initial_loss`. It memorises 8 real
RICE images, then runs the **canonical decode on the eval-mode path** over those
same images and requires AP50, AR@100 and train-vs-eval confidence parity.

Why the change: the 2026-07-28 checkpoints had a loss that fell for 14 straight
epochs and then decoded **AP 0.0000**. A falling loss was never evidence that the
detector could produce a box. The loss direction is still checked, but only as a
secondary condition.

The thresholds are **provisional smoke floors, not quality standards**
(CLAUDE.md section 38) -- they are set from pilot evidence, which is what this
notebook produces.

In [ ]:
# check=True: this is a gate. A failure must stop the notebook before two arms of
# GPU time are spent on a model that cannot decode a single box.
subprocess.run(
    ['python', '-B', '-u', '-m', 'agrinav.training.weeddet_train',
     '--ann-file', TRAIN_JSON, '--images-root', TRAIN_IMGS,
     '--class-names', 'rice_protect,weed_target',
     '--overfit', '8', '--batch-size', '2', '--img-size', '512',
     '--no-pretrained-backbone'],
    check=True)
print('overfit gate: PASSED (decoded AP50 + AR@100 + train/eval parity)')

## 7. Pilot configuration

Both arms share everything except what the backbone was loaded from and the BN
scope that follows from it. Same seed, same data, same schedule -- so a
difference between the arms is attributable.

In [ ]:
import datetime

EPOCHS      = 2
SEED        = 42
CLASS_NAMES = 'rice_protect,weed_target'
CONFIG      = 'configs/training/detector_rice_phase2.yaml'

TS         = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
PILOT_ROOT = f'{DRIVE_P2}/pilots/pilot_{TS}'
ARM_RICESEG  = f'{PILOT_ROOT}/riceseg'
ARM_IMAGENET = f'{PILOT_ROOT}/imagenet'
os.makedirs(PILOT_ROOT, exist_ok=True)

# Written to Drive so the pilot's identity survives the runtime, exactly as the
# full run's manifest does. A diagnostic whose provenance is unrecorded cannot be
# cited later as the reason a threshold was chosen.
def _git_out(*a):
    try:
        return subprocess.check_output(['git', '-C', '.', *a], text=True).strip()
    except Exception:
        return None

pilot_manifest = {
    'pilot_id': os.path.basename(PILOT_ROOT),
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'purpose': ('2-epoch instrumentation pilot: choose grad_clip, localise the '
                'train/eval BN gap, check the positive/negative loss split, and '
                'confirm the BN freeze held. NOT a training run and NOT a '
                'quality measurement -- 2 epochs decides nothing about accuracy.'),
    'git_commit': _git_out('rev-parse', 'HEAD'),
    'git_ref_requested': REPO_SHA or REPO_REF,
    'git_dirty': bool(_git_out('status', '--porcelain')),
    'epochs': EPOCHS,
    'seed': SEED,
    'config_file': CONFIG,
    'class_map': {n: i for i, n in enumerate(CLASS_NAMES.split(','))},
    'dataset_archive': zip_path,
    'dataset_archive_sha256': ARCHIVE_SHA256,
    'backbone_init': BACKBONE,
    'backbone_sha256': _sha256(BACKBONE),
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'arms': {
        'riceseg':  {'dir': ARM_RICESEG,  'bn_freeze_scope': 'backbone',
                     'expected_bn_frozen': '57 of 58'},
        'imagenet': {'dir': ARM_IMAGENET, 'bn_freeze_scope': 'imagenet',
                     'expected_bn_frozen': '48 of 58'},
    },
}
with open(f'{PILOT_ROOT}/pilot_manifest.json', 'w') as f:
    json.dump(pilot_manifest, f, indent=2, sort_keys=True)
print('pilot:', PILOT_ROOT)
print(json.dumps(pilot_manifest, indent=2, sort_keys=True))

### 7a. Arm A -- RiceSEG backbone (`--bn-freeze-scope backbone`, expect 57 of 58 frozen)

`--bn-freeze-scope backbone` is a **fail-closed assertion**, not a hint. If the
backbone load fills fewer layers than the scope covers, this aborts and names the
layers rather than quietly freezing a subset -- which is exactly the bug that made
`freeze_pretrained` a silent no-op (0 of 58) until 2026-07-31.

In [ ]:
arm_a = subprocess.run(
    ['python', '-B', '-u', '-m', 'agrinav.training.weeddet_train',
     '--config', CONFIG,
     '--ann-file', TRAIN_JSON, '--images-root', TRAIN_IMGS,
     '--val-ann-file', VAL_JSON, '--val-images-root', VAL_IMGS,
     '--class-names', CLASS_NAMES,
     '--epochs', str(EPOCHS), '--seed', str(SEED),
     '--riceseg-backbone', BACKBONE,
     '--bn-policy', 'freeze_pretrained', '--bn-freeze-scope', 'backbone',
     '--dump-grad-norms',
     '--checkpoint-dir', ARM_RICESEG],
    check=False)
print('arm A (riceseg) exit:', arm_a.returncode)
# Not check=True: arm B is still worth running if arm A fails, and the readout
# cell reports what each arm did or did not produce.

### 7b. Arm B -- ImageNet control (`--bn-freeze-scope imagenet`, expect 48 of 58 frozen)

In [ ]:
arm_b = subprocess.run(
    ['python', '-B', '-u', '-m', 'agrinav.training.weeddet_train',
     '--config', CONFIG,
     '--ann-file', TRAIN_JSON, '--images-root', TRAIN_IMGS,
     '--val-ann-file', VAL_JSON, '--val-images-root', VAL_IMGS,
     '--class-names', CLASS_NAMES,
     '--epochs', str(EPOCHS), '--seed', str(SEED),
     '--bn-policy', 'freeze_pretrained', '--bn-freeze-scope', 'imagenet',
     '--dump-grad-norms',
     '--checkpoint-dir', ARM_IMAGENET],
    check=False)
print('arm B (imagenet) exit:', arm_b.returncode)

## 8. Readout -- the four decisions

All analysis lives in `agrinav.training.pilot_report`, not in this cell, so the
same verdict can be reproduced from the run directories later without the
notebook. It exits non-zero if either arm raises a warning.

In [ ]:
from agrinav.training.pilot_report import format_report, summarise_arm

arms = []
for label, path in (('riceseg', ARM_RICESEG), ('imagenet', ARM_IMAGENET)):
    try:
        arms.append(summarise_arm(path, name=label))
    except FileNotFoundError as exc:
        print(f'!! {label}: {exc}')

report = format_report(arms) if arms else 'no arm produced a metrics.jsonl.'
print(report)

with open(f'{PILOT_ROOT}/pilot_readout.txt', 'w') as f:
    f.write(report + '\n')
print('\nsaved:', f'{PILOT_ROOT}/pilot_readout.txt')

### 8a. Full per-step gradient norms (optional)

`--dump-grad-norms` wrote every pre-clip step norm, not just the quantiles. Plot
it when the summary is ambiguous -- for example when p99 is moderate but `max` is
far away, which is a few violent steps rather than a heavy tail, and calls for a
different `grad_clip` than the quantile alone suggests.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, (label, path) in zip(axes, (('riceseg', ARM_RICESEG), ('imagenet', ARM_IMAGENET))):
    files = sorted(glob.glob(f'{path}/grad_norms_epoch*.json'))
    if not files:
        ax.set_title(f'{label}: no dump found')
        continue
    norms = []
    for fp in files:
        with open(fp) as fh:
            norms.extend(json.load(fh))
    ax.plot(norms, linewidth=0.5)
    ax.set_yscale('log')
    ax.set_title(f'{label}: {len(norms)} steps')
    ax.set_xlabel('step')
    for q, style in ((50, ':'), (99, '--')):
        v = float(torch.tensor(norms).quantile(q / 100))
        ax.axhline(v, linestyle=style, linewidth=1, label=f'p{q}={v:.3f}')
    ax.legend(fontsize=8)
axes[0].set_ylabel('pre-clip grad norm (log)')
plt.tight_layout()
plt.show()

## 9. What to do with the result

* **`grad_clip` BINDING** -- the clip, not the schedule, set every step. Raise it
  to the reported value or drop it, then update
  `configs/training/detector_rice_phase2.yaml` and record the change in
  `docs/GATE_STATUS.md` alongside the number that justified it.
* **A train/eval BN gap** -- the remaining trainable head BN (`head.shared.2.seq.3`)
  is the first suspect. The GroupNorm swap is the ablation, and it is deliberately
  not pre-emptive: it changes the `state_dict` and breaks every existing
  checkpoint, so it needs this evidence first.
* **`cls_loss_pos` flat while `cls_loss_neg` falls** -- the detector is learning
  "background". Do not spend 18 epochs on it; that is a loss or sampling problem.
* **All four clean** -- the 18-epoch run is unblocked. Update the row in
  `docs/GATE_STATUS.md` and run the phase-2 notebook.

Two epochs decide nothing about accuracy. Do not read AP from this pilot; that is
what the full run plus a same-protocol baseline are for, and **no baseline has
been run yet** (`docs/baselines.md`).